# Validation

In [1]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd

In [2]:
# Caminhos
xml_folder = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/raw/sim/ano_2022_xml/"
parquet_folder = "/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/sim/ano_2022/"

In [3]:
def analyze_xml(file_path, tag="row"):
    context = ET.iterparse(file_path, events=("end",))
    _, root = next(context)

    total_rows = 0
    columns = set()
    null_counts = defaultdict(int)

    for event, elem in context:
        if elem.tag == tag:
            total_rows += 1

            for child in elem:
                columns.add(child.tag)

                if child.text is None or child.text.strip() == "":
                    null_counts[child.tag] += 1

            # libera memória
            root.clear()

    return {
        "file": os.path.basename(file_path),
        "rows": total_rows,
        "columns": len(columns),
        "column_names": list(columns),
        "nulls": dict(null_counts)
    }

In [4]:
results = []

#for file in os.listdir(xml_folder):
for file in sorted(os.listdir(xml_folder)):
    if file.endswith(".xml"):
        file_path = os.path.join(xml_folder, file)
        print(f"Processando: {file}")

        result = analyze_xml(file_path)
        results.append(result)

Processando: DO22OPEN_00001.xml
Processando: DO22OPEN_00002.xml
Processando: DO22OPEN_00003.xml
Processando: DO22OPEN_00004.xml
Processando: DO22OPEN_00005.xml
Processando: DO22OPEN_00006.xml
Processando: DO22OPEN_00007.xml
Processando: DO22OPEN_00008.xml


In [4]:
# print(results[0]["column_names"])

['CODMUNRES', 'ESCMAEAGR1', 'ORIGEM', 'ACIDTRAB', 'OBITOGRAV', 'ESCMAE', 'IDADEMAE', 'CB_PRE', 'FONTES', 'NUDIASINF', 'DTRECORIGA', 'NUDIASOBIN', 'RACACOR', 'TPOBITOCOR', 'SERIESCFAL', 'CIRURGIA', 'DIFDATA', 'DTCADINF', 'SEXO', 'IDADE', 'FONTEINV', 'GRAVIDEZ', 'DTCADINV', 'NECROPSIA', 'STDONOVA', 'DTRECEBIM', 'ALTCAUSA', 'STDOEPIDEM', 'DTATESTADO', 'COMUNSVOIM', 'ASSISTMED', 'ESCMAE2010', 'DTCONINV', 'ESCFALAGR1', 'PESO', 'QTDFILMORT', 'TPPOS', 'CAUSABAS_O', 'TPRESGINFO', 'DTCADASTRO', 'CODMUNOCOR', 'OCUP', 'LINHAII', 'FONTE', 'NUDIASOBCO', 'DTINVESTIG', 'NUMEROLOTE', 'FONTESINF', 'TIPOBITO', 'OBITOPARTO', 'ESTCIV', 'SERIESCMAE', 'CONTADOR', 'LINHAA', 'MORTEPARTO', 'GESTACAO', 'NATURAL', 'ESC', 'CIRCOBITO', 'ATESTANTE', 'index', 'OCUPMAE', 'LINHAC', 'VERSAOSCB', 'TPMORTEOCO', 'LINHAB', 'CODESTAB', 'LINHAD', 'VERSAOSIST', 'ATESTADO', 'CODMUNNATU', 'PARTO', 'CODIFICADO', 'TPNIVELINV', 'DTOBITO', 'OBITOPUERP', 'CAUSABAS', 'STCODIFICA', 'ESC2010', 'CAUSAMAT', 'LOCOCOR', 'SEMAGESTAC', 'HORA

In [5]:
def analyze_parquet(parquet_folder):
    df = pd.read_parquet(parquet_folder)

    return {
        "file": os.path.basename(parquet_folder),
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": list(df.columns),
        "nulls": df.isnull().sum().to_dict()
    }

In [6]:
import re

# Ordenar numericamente os XMLs corretamente
def extract_xml_number(filename):
    return int(re.search(r'_(\d+)\.xml', filename).group(1))

xml_files = sorted(
    [f for f in os.listdir(xml_folder) if f.endswith(".xml")],
    key=extract_xml_number
)

In [7]:
# Ordenar numericamente os parquets
def extract_part_number(filename):
    return int(re.search(r'part_(\d+)\.parquet', filename).group(1))

parquet_files = sorted(
    [f for f in os.listdir(parquet_folder) if f.endswith(".parquet")],
    key=extract_part_number
)

In [8]:
print(xml_files)
print(parquet_files)

['DO22OPEN_00001.xml', 'DO22OPEN_00002.xml', 'DO22OPEN_00003.xml', 'DO22OPEN_00004.xml', 'DO22OPEN_00005.xml', 'DO22OPEN_00006.xml', 'DO22OPEN_00007.xml', 'DO22OPEN_00008.xml']
['part_0.parquet', 'part_1.parquet', 'part_2.parquet', 'part_3.parquet', 'part_4.parquet', 'part_5.parquet', 'part_6.parquet', 'part_7.parquet']


In [9]:
comparison_results = []

for xml_file, pq_file in zip(xml_files, parquet_files):
    
    xml_path = os.path.join(xml_folder, xml_file)
    pq_path = os.path.join(parquet_folder, pq_file)

    print(f"Comparando: {xml_file} ↔ {pq_file}")

    xml_stats = analyze_xml(xml_path)
    pq_stats = analyze_parquet(pq_path)

    comparison = {
        "xml_file": xml_file,
        "pq_file": pq_file,

        "xml_rows": xml_stats["rows"],
        "pq_rows": pq_stats["rows"],
        "row_diff": xml_stats["rows"] - pq_stats["rows"],

        "xml_cols": xml_stats["columns"],
        "pq_cols": pq_stats["columns"],
        "col_diff": xml_stats["columns"] - pq_stats["columns"],
    }

    comparison_results.append(comparison)

Comparando: DO22OPEN_00001.xml ↔ part_0.parquet
Comparando: DO22OPEN_00002.xml ↔ part_1.parquet
Comparando: DO22OPEN_00003.xml ↔ part_2.parquet
Comparando: DO22OPEN_00004.xml ↔ part_3.parquet
Comparando: DO22OPEN_00005.xml ↔ part_4.parquet
Comparando: DO22OPEN_00006.xml ↔ part_5.parquet
Comparando: DO22OPEN_00007.xml ↔ part_6.parquet
Comparando: DO22OPEN_00008.xml ↔ part_7.parquet


- 2018

In [10]:
df_compare = pd.DataFrame(comparison_results)
df_compare

,xml_file,pq_file,xml_rows,pq_rows,row_diff,xml_cols,pq_cols,col_diff
0,Mortalidade_Geral_2018_00001.xml,part_0.parquet,200000,200000,0,88,88,0
1,Mortalidade_Geral_2018_00002.xml,part_1.parquet,200000,200000,0,88,88,0
2,Mortalidade_Geral_2018_00003.xml,part_2.parquet,200000,200000,0,88,88,0
3,Mortalidade_Geral_2018_00004.xml,part_3.parquet,200000,200000,0,88,88,0
4,Mortalidade_Geral_2018_00005.xml,part_4.parquet,200000,200000,0,88,88,0
5,Mortalidade_Geral_2018_00006.xml,part_5.parquet,200000,200000,0,88,88,0
6,Mortalidade_Geral_2018_00007.xml,part_6.parquet,116719,116719,0,88,88,0


- 2019

In [10]:
df_compare = pd.DataFrame(comparison_results)
df_compare

,xml_file,pq_file,xml_rows,pq_rows,row_diff,xml_cols,pq_cols,col_diff
0,Mortalidade_Geral_2019_00001.xml,part_0.parquet,200000,200000,0,88,88,0
1,Mortalidade_Geral_2019_00002.xml,part_1.parquet,200000,200000,0,88,88,0
2,Mortalidade_Geral_2019_00003.xml,part_2.parquet,200000,200000,0,88,88,0
3,Mortalidade_Geral_2019_00004.xml,part_3.parquet,200000,200000,0,88,88,0
4,Mortalidade_Geral_2019_00005.xml,part_4.parquet,200000,200000,0,88,88,0
5,Mortalidade_Geral_2019_00006.xml,part_5.parquet,200000,200000,0,88,88,0
6,Mortalidade_Geral_2019_00007.xml,part_6.parquet,149801,149801,0,88,88,0


- 2020

In [19]:
df_compare = pd.DataFrame(comparison_results)
df_compare

,xml_file,pq_file,xml_rows,pq_rows,row_diff,xml_cols,pq_cols,col_diff
0,Mortalidade_Geral_2020_00001.xml,part_0.parquet,200000,200000,0,88,88,0
1,Mortalidade_Geral_2020_00002.xml,part_1.parquet,200000,200000,0,88,88,0
2,Mortalidade_Geral_2020_00003.xml,part_2.parquet,200000,200000,0,88,88,0
3,Mortalidade_Geral_2020_00004.xml,part_3.parquet,200000,200000,0,88,88,0
4,Mortalidade_Geral_2020_00005.xml,part_4.parquet,200000,200000,0,88,88,0
5,Mortalidade_Geral_2020_00006.xml,part_5.parquet,200000,200000,0,88,88,0
6,Mortalidade_Geral_2020_00007.xml,part_6.parquet,200000,200000,0,88,88,0
7,Mortalidade_Geral_2020_00008.xml,part_7.parquet,156824,156824,0,88,88,0


- 2021

In [10]:
df_compare = pd.DataFrame(comparison_results)
df_compare

,xml_file,pq_file,xml_rows,pq_rows,row_diff,xml_cols,pq_cols,col_diff
0,Mortalidade_Geral_2021_00001.xml,part_0.parquet,200000,200000,0,88,88,0
1,Mortalidade_Geral_2021_00002.xml,part_1.parquet,200000,200000,0,88,88,0
2,Mortalidade_Geral_2021_00003.xml,part_2.parquet,200000,200000,0,88,88,0
3,Mortalidade_Geral_2021_00004.xml,part_3.parquet,200000,200000,0,88,88,0
4,Mortalidade_Geral_2021_00005.xml,part_4.parquet,200000,200000,0,88,88,0
5,Mortalidade_Geral_2021_00006.xml,part_5.parquet,200000,200000,0,88,88,0
6,Mortalidade_Geral_2021_00007.xml,part_6.parquet,200000,200000,0,88,88,0
7,Mortalidade_Geral_2021_00008.xml,part_7.parquet,200000,200000,0,88,88,0
8,Mortalidade_Geral_2021_00009.xml,part_8.parquet,200000,200000,0,88,88,0
9,Mortalidade_Geral_2021_00010.xml,part_9.parquet,32649,32649,0,88,88,0


- 2022

In [10]:
df_compare = pd.DataFrame(comparison_results)
df_compare

,xml_file,pq_file,xml_rows,pq_rows,row_diff,xml_cols,pq_cols,col_diff
0,DO22OPEN_00001.xml,part_0.parquet,200000,200000,0,88,88,0
1,DO22OPEN_00002.xml,part_1.parquet,200000,200000,0,88,88,0
2,DO22OPEN_00003.xml,part_2.parquet,200000,200000,0,88,88,0
3,DO22OPEN_00004.xml,part_3.parquet,200000,200000,0,88,88,0
4,DO22OPEN_00005.xml,part_4.parquet,200000,200000,0,88,88,0
5,DO22OPEN_00006.xml,part_5.parquet,200000,200000,0,88,88,0
6,DO22OPEN_00007.xml,part_6.parquet,200000,200000,0,88,88,0
7,DO22OPEN_00008.xml,part_7.parquet,144266,144266,0,88,88,0
